In [0]:
from pyspark.sql.functions import col, window, sum as _sum, count, approx_count_distinct

# ---- Windowed aggregation with watermark ----
gold_stream = (spark.readStream
    .table("eh_streaming.oms.silver_orders")
    .withWatermark("event_ts", "10 minutes")              # tolerate up to 10 min of late events
    .groupBy(
        window(col("event_ts"), "1 minute"),              # tumbling 1-minute windows on event time
        col("product_category"),
        col("country"),
    )
    .agg(
        _sum("revenue").alias("revenue"),
        count("order_id").alias("order_count"),
        approx_count_distinct("user_id").alias("unique_users"),
    )
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "product_category",
        "country",
        "revenue",
        "order_count",
        "unique_users",
    )
)

gold_stream.printSchema()

root
 |-- window_start: timestamp (nullable = true)
 |-- window_end: timestamp (nullable = true)
 |-- product_category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- revenue: double (nullable = true)
 |-- order_count: long (nullable = false)
 |-- unique_users: long (nullable = false)



In [0]:
# ---- Write to Gold Delta table ----
(gold_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/eh_streaming/oms/checkpoints/gold")
    .trigger(processingTime="30 seconds")
    .toTable("eh_streaming.oms.gold_revenue_by_cat_country_1m"))

In [0]:
# ---- Validate ----
print("Gold count:", spark.read.table("eh_streaming.oms.gold_revenue_by_cat_country_1m").count())
display(
    spark.read.table("eh_streaming.oms.gold_revenue_by_cat_country_1m")
        .orderBy("window_start", "product_category", "country")
        .limit(20)
)

Gold count: 360


window_start,window_end,product_category,country,revenue,order_count,unique_users
2026-05-27T04:45:00Z,2026-05-27T04:46:00Z,books,BR,3378.309999999999,57,59
2026-05-27T04:45:00Z,2026-05-27T04:46:00Z,books,CA,4637.679999999997,79,76
2026-05-27T04:45:00Z,2026-05-27T04:46:00Z,books,DE,3538.2299999999987,60,62
2026-05-27T04:45:00Z,2026-05-27T04:46:00Z,books,FR,2338.83,47,48
2026-05-27T04:45:00Z,2026-05-27T04:46:00Z,books,GB,3278.3599999999983,56,55
2026-05-27T04:45:00Z,2026-05-27T04:46:00Z,books,IN,3638.179999999998,60,59
2026-05-27T04:45:00Z,2026-05-27T04:46:00Z,books,JP,3518.239999999997,62,62
2026-05-27T04:45:00Z,2026-05-27T04:46:00Z,books,US,2998.4999999999986,47,48
2026-05-27T04:45:00Z,2026-05-27T04:46:00Z,clothing,BR,24259.93999999999,121,123
2026-05-27T04:45:00Z,2026-05-27T04:46:00Z,clothing,CA,24597.820000000007,113,116


In [0]:

silver_totals = spark.sql("""
    SELECT SUM(revenue) AS total_revenue, COUNT(*) AS total_orders
    FROM eh_streaming.oms.silver_orders
""")

gold_totals = spark.sql("""
    SELECT SUM(revenue) AS total_revenue, SUM(order_count) AS total_orders
    FROM eh_streaming.oms.gold_revenue_by_cat_country_1m
""")

display(silver_totals)
display(gold_totals)

total_revenue,total_orders
4.8492092100000605E7,56966


total_revenue,total_orders
2.7098034849999998E7,31811


In [0]:
%sql

SELECT window_start, product_category, SUM(revenue) AS revenue
FROM eh_streaming.oms.gold_revenue_by_cat_country_1m
GROUP BY window_start, product_category
ORDER BY window_start

window_start,product_category,revenue
2026-05-27T04:45:00Z,clothing,188202.61000000002
2026-05-27T04:45:00Z,books,27326.329999999987
2026-05-27T04:45:00Z,sports,107250.0
2026-05-27T04:45:00Z,home,212742.0
2026-05-27T04:45:00Z,electronics,2154346.2100000004
2026-05-27T04:46:00Z,sports,108150.0
2026-05-27T04:46:00Z,books,30384.799999999974
2026-05-27T04:46:00Z,clothing,220772.32000000004
2026-05-27T04:46:00Z,home,232935.0
2026-05-27T04:46:00Z,electronics,2536312.4799999995


Databricks visualization. Run in Databricks to view.

In [0]:
for q in spark.streams.active:
    print(f"Stopping {q.name} ({q.id})")
    q.stop()
print("All streams stopped.")

Stopping  (d7668c9e-1572-47b3-a157-dbd18b0888d7)
All streams stopped.
